<a href="https://colab.research.google.com/github/rkabishi/michael/blob/master/TwitterSentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle

In [2]:
# configure the path to kaggle.json file

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


# IMPORTING TWITTER DATASET FROM KAGGLE USING API

In [3]:
# API to get the dataset from kaggle
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
 98% 79.0M/80.9M [00:00<00:00, 213MB/s]
100% 80.9M/80.9M [00:00<00:00, 205MB/s]


In [4]:
# Zip file dataset extraction
from zipfile import ZipFile
dataset = '/content/sentiment140.zip'

with ZipFile(dataset,'r') as zip:
  zip.extractall()
  print('The dataset is extracted')


The dataset is extracted


In [5]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [6]:
import nltk
nltk.download('stopwords')
print(stopwords.words('english'))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# DATA PROCESSING

In [7]:
# LOADING DATA FROM CSV FILE TO PANDAS DATAFRAME

# Loading data from csv file to pandas dataframe
df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',encoding='ISO-8859-1',header=None)
df.columns = ['target','id','date','flag','user','text']
df.shape
df.head()


,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [8]:
# counting the missing values in the dataset

df.isnull().sum()


,0
target,0
id,0
date,0
flag,0
user,0
text,0


In [9]:
# checking the target column distribution

df['target'].value_counts()


,count
target,
0,800000
4,800000


In [10]:
# converting the target "4" to "1 "

df['target'] = df['target'].replace(4,1)


In [11]:
# "0" means negative tweet and "1" is positive tweet
df['target'].value_counts()

,count
target,
0,800000
1,800000


## Stemming (reducing words to its root or key word)

In [12]:
# write stemming function using PorterStemmer and re

stemmer = PorterStemmer()
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [stemmer.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content


In [13]:
# Apply the stemming function to our data text column  and name it stemmed_text

df['stemmed_text'] = df['text'].apply(stemming)


In [14]:
# print the stemmed content and target

print(df[['stemmed_text','target']])


                                              stemmed_text  target
0        switchfoot http twitpic com zl awww bummer sho...       0
1        upset updat facebook text might cri result sch...       0
2        kenichan dive mani time ball manag save rest g...       0
3                          whole bodi feel itchi like fire       0
4                            nationwideclass behav mad see       0
...                                                    ...     ...
1599995                         woke school best feel ever       1
1599996  thewdb com cool hear old walt interview http b...       1
1599997                       readi mojo makeov ask detail       1
1599998  happi th birthday boo alll time tupac amaru sh...       1
1599999  happi charitytuesday thenspcc sparkschar speak...       1

[1600000 rows x 2 columns]


In [15]:
# Separate the data and label

X = df['stemmed_text'].values
y = df['target'].values


## Spliting data into train and test

In [16]:
# Split the data into train ,test

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=2)


In [17]:
print(X.shape,X_train.shape,X_test.shape)

(1600000,) (1280000,) (320000,)


## Convert text data to numerical data

In [18]:

# converting the textual data to numerical data
vectorizer = CountVectorizer()
vectorizer.fit(X_train)

X_train = vectorizer.transform(X_train)
X_test = vectorizer.transform(X_test)

print(X_train)


  (0, 31233)	1
  (0, 129956)	1
  (1, 241525)	1
  (1, 437910)	1
  (2, 42838)	1
  (2, 93639)	1
  (2, 128288)	1
  (2, 140234)	1
  (2, 145727)	1
  (2, 237072)	1
  (2, 278606)	1
  (2, 411086)	1
  (2, 416219)	1
  (3, 8560)	1
  (3, 179097)	1
  (3, 201928)	1
  (3, 286683)	1
  (3, 308910)	1
  (3, 445747)	1
  (4, 227660)	1
  (4, 302738)	1
  (4, 315347)	1
  (4, 415818)	1
  (4, 457632)	1
  (5, 93639)	1
  :	:
  (1279996, 443184)	1
  (1279996, 445242)	1
  (1279996, 452298)	1
  (1279997, 127887)	1
  (1279997, 135226)	1
  (1279997, 312157)	1
  (1279997, 336356)	1
  (1279997, 420583)	1
  (1279998, 20513)	1
  (1279998, 22579)	1
  (1279998, 86796)	1
  (1279998, 158430)	1
  (1279998, 234184)	1
  (1279998, 272146)	1
  (1279998, 274147)	1
  (1279998, 302738)	1
  (1279998, 354295)	1
  (1279999, 29166)	1
  (1279999, 67237)	1
  (1279999, 125365)	1
  (1279999, 156026)	1
  (1279999, 177712)	1
  (1279999, 267262)	1
  (1279999, 391288)	1
  (1279999, 411086)	1


In [19]:
print(X_train)

  (0, 31233)	1
  (0, 129956)	1
  (1, 241525)	1
  (1, 437910)	1
  (2, 42838)	1
  (2, 93639)	1
  (2, 128288)	1
  (2, 140234)	1
  (2, 145727)	1
  (2, 237072)	1
  (2, 278606)	1
  (2, 411086)	1
  (2, 416219)	1
  (3, 8560)	1
  (3, 179097)	1
  (3, 201928)	1
  (3, 286683)	1
  (3, 308910)	1
  (3, 445747)	1
  (4, 227660)	1
  (4, 302738)	1
  (4, 315347)	1
  (4, 415818)	1
  (4, 457632)	1
  (5, 93639)	1
  :	:
  (1279996, 443184)	1
  (1279996, 445242)	1
  (1279996, 452298)	1
  (1279997, 127887)	1
  (1279997, 135226)	1
  (1279997, 312157)	1
  (1279997, 336356)	1
  (1279997, 420583)	1
  (1279998, 20513)	1
  (1279998, 22579)	1
  (1279998, 86796)	1
  (1279998, 158430)	1
  (1279998, 234184)	1
  (1279998, 272146)	1
  (1279998, 274147)	1
  (1279998, 302738)	1
  (1279998, 354295)	1
  (1279999, 29166)	1
  (1279999, 67237)	1
  (1279999, 125365)	1
  (1279999, 156026)	1
  (1279999, 177712)	1
  (1279999, 267262)	1
  (1279999, 391288)	1
  (1279999, 411086)	1


In [20]:
print(X_test)

  (0, 5622)	1
  (0, 53069)	1
  (0, 149573)	1
  (0, 177712)	1
  (0, 220160)	1
  (0, 272353)	1
  (0, 280262)	1
  (0, 298811)	1
  (0, 313583)	1
  (0, 317487)	1
  (0, 334325)	1
  (0, 371823)	1
  (0, 387006)	1
  (0, 414670)	1
  (0, 428015)	1
  (0, 445807)	1
  (1, 30475)	1
  (1, 154446)	1
  (2, 39327)	1
  (2, 43578)	1
  (2, 306745)	1
  (2, 344198)	1
  (2, 384696)	1
  (3, 57543)	1
  (3, 218938)	1
  :	:
  (319995, 451190)	1
  (319996, 132075)	1
  (319996, 135226)	2
  (319996, 325461)	1
  (319996, 358019)	1
  (319996, 385819)	1
  (319996, 408740)	1
  (319996, 419143)	1
  (319997, 37607)	1
  (319997, 150554)	1
  (319997, 267262)	1
  (319997, 279598)	1
  (319998, 145727)	1
  (319998, 178301)	1
  (319998, 220160)	1
  (319998, 336003)	1
  (319999, 32895)	1
  (319999, 66930)	1
  (319999, 136155)	1
  (319999, 181950)	1
  (319999, 230791)	1
  (319999, 236732)	1
  (319999, 301415)	1
  (319999, 349279)	1
  (319999, 445242)	1


#TRAINING THE MODEL

In [21]:
#TRAIN THE MODEL

model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)


LogisticRegression(max_iter=1000)

In [22]:


# model evaluation

# accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, y_train)
print('Accuracy score of the training data : ', training_data_accuracy)

# accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, y_test)
print('Accuracy score of the test data : ', test_data_accuracy)


Accuracy score of the training data :  0.83016015625
Accuracy score of the test data :  0.775721875


## Saving the trained model

In [23]:
# save this trained model

import pickle
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))


In [24]:
# use the saved model for future prediction

# loading the saved model
loaded_model = pickle.load(open('trained_model.sav', 'rb'))

# make prediction
input_data = ["I love this product!"]
input_data = vectorizer.transform(input_data)
prediction = loaded_model.predict(input_data)
print(prediction)


[1]


In [25]:


# use the saved model for future prediction with longer tweet

# loading the saved model
loaded_model = pickle.load(open('trained_model.sav', 'rb'))

# make prediction
input_data = ["I am so happy with this product! It exceeded my expectations and I would definitely recommend it to others."]
input_data = vectorizer.transform(input_data)
prediction = loaded_model.predict(input_data)
print(prediction)


[1]


In [26]:


# use the saved model for future prediction with longer tweet

# loading the saved model
loaded_model = pickle.load(open('trained_model.sav', 'rb'))

# make prediction
input_data = ["This is the worst product ever. I am so disappointed."]
input_data = vectorizer.transform(input_data)
prediction = loaded_model.predict(input_data)
print(prediction)


[0]
